<a href="https://colab.research.google.com/github/RodrigoCasanova/Backend/blob/main/BIY7121_Entrega_3_G1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto semestral - Fase 3 - BIY7121 - Mineria de Datos
## Evaluacion 3 - Preparacion de datos y modelos de clasificacion y segmentacion

**Grupo 1**

Autor: Rodrigo Casanova

Fecha de creacion: Junio de 2026
Version: 1.0

## Descripcion
Este notebook desarrolla la fase de modelamiento solicitada en la evaluacion.
Primero se construyen tres modelos de clasificacion usando solo transformacion ciclica y luego se desarrolla la segmentacion con K-Means.


## Importacion de librerias
Se utilizan las librerias necesarias para preparacion de datos, modelamiento, evaluacion y visualizacion.


In [ ]:
!pip install kneed -q


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from kneed import KneeLocator


## Carga de datos
El conjunto de datos corresponde al archivo climatico usado en la evaluacion.
Si la ruta cambia, basta con modificar la variable `url`.


In [ ]:
url = 'https://raw.githubusercontent.com/RodrigoCasanova/Mineria_de_datos_grupo1/refs/heads/main/datos/data_clima_2025_final.csv'
data = pd.read_csv(url, sep=',', low_memory=False)
data.columns = data.columns.str.strip().str.lower()
target = 'is_rainy_hour'
data.head()


### Revision inicial
Se revisa la estructura general del conjunto de datos para entender su forma y tipos de variables.


In [ ]:
data.shape


In [ ]:
data.dtypes


### Distribucion de la variable objetivo
Se observa la proporcion de clases de `is_rainy_hour` para tener una primera idea del desbalance.


In [ ]:
data[target].value_counts().sort_index().plot(kind='pie', autopct='%1.1f%%', labels=['No lluvia', 'Lluvia'], figsize=(6, 6))
plt.title('Distribucion de la variable objetivo', fontsize=14, fontweight='bold')
plt.ylabel('')
plt.show()


## Fase 2 - Entendimiento de los datos
### Transformacion ciclica
Segun la pauta, para la clasificacion se trabaja solo con transformacion ciclica sobre el tiempo.
No se crean rezagos ni variables de futuro.


In [ ]:
def crear_variables_ciclicas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['fecha'] = pd.to_datetime(df['date'])
    df['hora'] = df['fecha'].dt.hour
    df['mes'] = df['fecha'].dt.month

    df['hora_sin'] = np.sin(2 * np.pi * df['hora'] / 24)
    df['hora_cos'] = np.cos(2 * np.pi * df['hora'] / 24)
    df['mes_sin'] = np.sin(2 * np.pi * df['mes'] / 12)
    df['mes_cos'] = np.cos(2 * np.pi * df['mes'] / 12)

    df = df.drop(columns=['date', 'fecha', 'hora', 'mes'])
    return df

data = crear_variables_ciclicas(data)
data.head()


### Variables usadas en clasificacion
Se eligen variables climaticas disponibles al momento de predecir, junto con las variables ciclicas de tiempo.
Tambien se incorpora `localidad`, porque ayuda a entender mejor el contexto geografico.


In [ ]:
features_clasificacion = [
    'temperature_2m',
    'relative_humidity_2m',
    'apparent_temperature',
    'cloud_cover',
    'wind_speed_10m',
    'sunshine_duration',
    'localidad',
    'is_day',
    'hora_sin',
    'hora_cos',
    'mes_sin',
    'mes_cos',
]

X = data[features_clasificacion].copy()
y = data[target].copy()
X.head()


### Comprobacion del supuesto de GaussianNB
GaussianNB asume independencia condicional aproximada entre variables dado el valor de la clase.
Para revisarlo, se compara la correlacion entre variables en el conjunto completo y luego por clase.


In [ ]:
variables_revision = [
    'temperature_2m',
    'relative_humidity_2m',
    'cloud_cover',
    'wind_speed_10m',
    'sunshine_duration',
    'hora_sin',
    'hora_cos',
    'mes_sin',
    'mes_cos',
]

corr_general = data[variables_revision].corr()
display(corr_general.round(2))


In [ ]:
for clase in sorted(data[target].dropna().unique()):
    print(f'Correlaciones condicionadas a la clase {clase}')
    display(data.loc[data[target] == clase, variables_revision].corr().round(2))


Interpretacion:
Si las correlaciones dentro de cada clase disminuyen respecto de las correlaciones del conjunto total,
entonces el supuesto de independencia condicional de Naive Bayes es razonable para este problema.


### Separacion de entrenamiento y prueba
Se utiliza una particion estratificada para mantener la proporcion de clases en ambos conjuntos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=29,
    stratify=y,
)


### Preprocesamiento
Se imputan y estandarizan las variables numericas, y se codifica la variable categorica `localidad`.


In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

features_numericas = [
    'temperature_2m',
    'relative_humidity_2m',
    'apparent_temperature',
    'cloud_cover',
    'wind_speed_10m',
    'sunshine_duration',
    'is_day',
    'hora_sin',
    'hora_cos',
    'mes_sin',
    'mes_cos',
]

features_categoricas = ['localidad']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, features_numericas),
        ('cat', categorical_transformer, features_categoricas),
    ],
    remainder='drop'
)


### Funcion de evaluacion
La funcion retorna las metricas principales para comparar los tres modelos de manera uniforme.


In [ ]:
def evaluar_modelo(modelo, X_train, y_train, X_test, y_test):
    y_pred_train = modelo.predict(X_train)
    y_prob_train = modelo.predict_proba(X_train)[:, 1]
    y_pred_test = modelo.predict(X_test)
    y_prob_test = modelo.predict_proba(X_test)[:, 1]

    return {
        'accuracy_train': accuracy_score(y_train, y_pred_train),
        'accuracy_test': accuracy_score(y_test, y_pred_test),
        'precision_train': precision_score(y_train, y_pred_train, zero_division=0),
        'precision_test': precision_score(y_test, y_pred_test, zero_division=0),
        'recall_train': recall_score(y_train, y_pred_train, zero_division=0),
        'recall_test': recall_score(y_test, y_pred_test, zero_division=0),
        'f1_train': f1_score(y_train, y_pred_train, zero_division=0),
        'f1_test': f1_score(y_test, y_pred_test, zero_division=0),
        'roc_auc_train': roc_auc_score(y_train, y_prob_train),
        'roc_auc_test': roc_auc_score(y_test, y_prob_test),
        'y_pred_test': y_pred_test,
        'y_prob_test': y_prob_test,
    }


## Fase 4 - Modelamiento
### Modelos de clasificacion
Se construyen LogisticRegression, DecisionTreeClassifier y GaussianNB usando GridSearchCV.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=29)


In [ ]:
pipeline_lr = Pipeline(steps=[
    ('preprocesador', preprocessor),
    ('modelo', LogisticRegression(max_iter=1000))
])

param_grid_lr = {
    'modelo__C': [0.1, 1, 10],
    'modelo__solver': ['liblinear'],
    'modelo__class_weight': [None, 'balanced'],
}

grid_lr = GridSearchCV(
    estimator=pipeline_lr,
    param_grid=param_grid_lr,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
)
grid_lr.fit(X_train, y_train)
grid_lr.best_params_


In [ ]:
pipeline_dtc = Pipeline(steps=[
    ('preprocesador', preprocessor),
    ('modelo', DecisionTreeClassifier(random_state=29))
])

param_grid_dtc = {
    'modelo__max_depth': [3, 5, 8],
    'modelo__min_samples_split': [2, 5, 10],
    'modelo__min_samples_leaf': [1, 3, 5],
    'modelo__class_weight': [None, 'balanced'],
}

grid_dtc = GridSearchCV(
    estimator=pipeline_dtc,
    param_grid=param_grid_dtc,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
)
grid_dtc.fit(X_train, y_train)
grid_dtc.best_params_


In [ ]:
pipeline_nb = Pipeline(steps=[
    ('preprocesador', preprocessor),
    ('modelo', GaussianNB())
])

param_grid_nb = {
    'modelo__var_smoothing': np.logspace(-12, -7, 8)
}

grid_nb = GridSearchCV(
    estimator=pipeline_nb,
    param_grid=param_grid_nb,
    cv=cv,
    scoring='f1',
    n_jobs=-1,
)
grid_nb.fit(X_train, y_train)
grid_nb.best_params_


### Seleccion automatica del mejor modelo
Se compara el desempeno en prueba de los tres modelos ajustados y se elige automaticamente el mejor segun F1.


In [ ]:
mejores_modelos = {
    'LogisticRegression': grid_lr.best_estimator_,
    'DecisionTreeClassifier': grid_dtc.best_estimator_,
    'GaussianNB': grid_nb.best_estimator_,
}

resultados_modelos = []
for nombre, modelo in mejores_modelos.items():
    metricas = evaluar_modelo(modelo, X_train, y_train, X_test, y_test)
    resultados_modelos.append({
        'modelo': nombre,
        'accuracy_train': metricas['accuracy_train'],
        'accuracy_test': metricas['accuracy_test'],
        'precision_train': metricas['precision_train'],
        'precision_test': metricas['precision_test'],
        'recall_train': metricas['recall_train'],
        'recall_test': metricas['recall_test'],
        'f1_train': metricas['f1_train'],
        'f1_test': metricas['f1_test'],
        'roc_auc_train': metricas['roc_auc_train'],
        'roc_auc_test': metricas['roc_auc_test'],
        'diff_f1': abs(metricas['f1_train'] - metricas['f1_test']),
        'diff_roc_auc': abs(metricas['roc_auc_train'] - metricas['roc_auc_test']),
    })

tabla_resultados = pd.DataFrame(resultados_modelos).sort_values(
    by=['f1_test', 'roc_auc_test'], ascending=False
).reset_index(drop=True)
tabla_resultados


In [ ]:
nombre_mejor_modelo = tabla_resultados.loc[0, 'modelo']
modelo_ganador = mejores_modelos[nombre_mejor_modelo]
print(f'Mejor modelo seleccionado: {nombre_mejor_modelo}')


### Matriz de confusion y metricas del mejor modelo
Se usan las predicciones con umbral 0.5, que corresponde al valor por defecto.


In [ ]:
metricas_ganador = evaluar_modelo(modelo_ganador, X_train, y_train, X_test, y_test)

print(f"{'Accuracy':<20}: {metricas_ganador['accuracy_test']:.4f}")
print(f"{'Precision':<20}: {metricas_ganador['precision_test']:.4f}")
print(f"{'Recall':<20}: {metricas_ganador['recall_test']:.4f}")
print(f"{'F1-score':<20}: {metricas_ganador['f1_test']:.4f}")
print(f"{'ROC-AUC':<20}: {metricas_ganador['roc_auc_test']:.4f}")


In [ ]:
y_pred = modelo_ganador.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No lluvia', 'Lluvia'])
disp.plot(cmap='Blues')
plt.title('Matriz de confusion del mejor modelo', fontsize=14, fontweight='bold')
plt.xlabel('Prediccion')
plt.ylabel('Valor real')
plt.show()


Interpretacion:
La matriz de confusion permite ver cuantas veces el modelo acierta y cuantas veces se equivoca al predecir lluvia.
En contexto de negocio, un falso negativo significa no anticipar una hora lluviosa, mientras que un falso positivo significa alertar lluvia cuando no ocurrira.


### Ajuste de umbral
Se analizan los umbrales 0.3 y 0.7 para observar el cambio en precision, recall y F1.


In [ ]:
def evaluar_umbral(modelo, X_test, y_test, umbral, nombre_archivo):
    probabilidades = modelo.predict_proba(X_test)[:, 1]
    predicciones = (probabilidades >= umbral).astype(int)

    precision = precision_score(y_test, predicciones, zero_division=0)
    recall = recall_score(y_test, predicciones, zero_division=0)
    f1 = f1_score(y_test, predicciones, zero_division=0)
    cm = confusion_matrix(y_test, predicciones)

    print(f'Umbral: {umbral:.1f}')
    print(f"{'Precision':<20}: {precision:.4f}")
    print(f"{'Recall':<20}: {recall:.4f}")
    print(f"{'F1-score':<20}: {f1:.4f}")

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No lluvia', 'Lluvia'])
    disp.plot(cmap='Blues')
    plt.title(f'Matriz de confusion umbral {umbral:.1f}', fontsize=14, fontweight='bold')
    plt.xlabel('Prediccion')
    plt.ylabel('Valor real')
    plt.show()

    salida = X_test.copy()
    salida['valor_real'] = y_test.values
    salida['probabilidad'] = probabilidades
    salida['prediccion'] = predicciones
    salida['umbral'] = umbral
    salida.to_csv(nombre_archivo, index=False)

    return {
        'umbral': umbral,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'archivo': nombre_archivo,
    }


In [ ]:
resultados_threshold = []
for umbral in [0.3, 0.7]:
    nombre_archivo = f'predicciones_umbral_{str(umbral).replace(".", "_")}_G1.csv'
    resultado = evaluar_umbral(modelo_ganador, X_test, y_test, umbral, nombre_archivo)
    resultados_threshold.append(resultado)

pd.DataFrame(resultados_threshold)


### Analisis comparativo
Al bajar el umbral, el modelo tiende a predecir mas casos positivos, lo que suele subir el recall y aumentar los falsos positivos.
Al subir el umbral, el modelo se vuelve mas exigente para marcar lluvia, lo que normalmente mejora precision pero reduce recall.


### Decision de negocio
El umbral elegido debe balancear el costo de no anticipar lluvia con el costo de generar falsas alertas.
En un problema meteorologico, normalmente se prefiere un umbral que no castigue demasiado el recall.


### CSV final con la misma observacion
Se guarda una sola observacion de prueba con las probabilidades y predicciones de los tres modelos.


In [ ]:
muestra = X_test.iloc[[0]].copy()
pred_final = muestra.copy()
pred_final['valor_real'] = y_test.iloc[[0]].values

for nombre, modelo in mejores_modelos.items():
    proba = modelo.predict_proba(muestra)[:, 1][0]
    pred = int(proba >= 0.5)
    pred_final[f'probabilidad_{nombre}'] = proba
    pred_final[f'prediccion_{nombre}'] = pred

pred_final.to_csv('predicciones_G1.csv', index=False)
pred_final


### Interpretacion final de clasificacion
Se comparan los tres modelos y se reporta el mejor segun F1. Ademas, se analiza como cambian las metricas cuando se modifica el umbral.
Con esto queda completa la parte de clasificacion solicitada por la pauta.


## Fase de segmentacion
En esta segunda parte se trabaja con exactamente seis variables y se aplica K-Means.


### Seleccion de seis variables
Se eligen seis variables climaticas para caracterizar cada segmento desde una perspectiva de negocio.
Estas variables resumen temperatura, humedad, nubosidad, lluvia, viento y disponibilidad de sol.


In [ ]:
variables_segmentacion = [
    'temperature_2m',
    'relative_humidity_2m',
    'cloud_cover',
    'precipitation',
    'wind_speed_10m',
    'sunshine_duration',
]

data_segmentacion = data[variables_segmentacion].copy()
data_segmentacion.head()


### Justificacion de negocio
Estas seis variables permiten separar dias u horas con condiciones climaticas distintas.
Por ejemplo, combinan calor, humedad, nubosidad, lluvia, ventilacion y radiacion solar, que son utiles para describir patrones de clima.


### Estandarizacion
Antes de aplicar K-Means se estandarizan las variables, porque el algoritmo trabaja con distancia euclidiana.


In [ ]:
preprocessor_seg = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
])

X_seg = preprocessor_seg.fit_transform(data_segmentacion)


### Metodo del codo con Kneed
Se calcula la inercia para varios valores de K y se usa `KneeLocator` para encontrar automaticamente el valor sugerido.


In [ ]:
inertias = []
rango_k = range(2, 11)

for k in rango_k:
    km = KMeans(n_clusters=k, random_state=29, n_init=10)
    km.fit(X_seg)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(rango_k), inertias, marker='o')
plt.title('Metodo del codo', fontsize=14, fontweight='bold')
plt.xlabel('K')
plt.ylabel('Inercia')
plt.grid(True)
plt.show()


In [ ]:
knee = KneeLocator(rango_k, inertias, curve='convex', direction='decreasing')
k_optimo = knee.elbow if knee.elbow is not None else 4
print(f'K recomendado por Kneed: {k_optimo}')


### Entrenamiento final de K-Means
Con el valor sugerido de K se entrena el modelo final y se asigna un cluster a cada observacion.


In [ ]:
kmeans = KMeans(n_clusters=k_optimo, random_state=29, n_init=10)
clusters = kmeans.fit_predict(X_seg)
data_segmentacion['cluster'] = clusters
data_segmentacion.head()


### PCA para visualizacion
PCA reduce la dimension a dos componentes para visualizar los clusters de forma clara.
Ademas permite reportar cuanta varianza explica la representacion bidimensional.


In [ ]:
pca = PCA(n_components=2, random_state=29)
X_pca = pca.fit_transform(X_seg)
centroides_pca = pca.transform(kmeans.cluster_centers_)
varianza_explicada = pca.explained_variance_ratio_.sum()
print(f'Varianza explicada por las dos componentes: {varianza_explicada:.4f}')


In [ ]:
plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='tab10', alpha=0.7, s=25)
plt.scatter(centroides_pca[:, 0], centroides_pca[:, 1], c='black', s=250, marker='X', label='Centroides')
plt.title('Clusters proyectados con PCA', fontsize=14, fontweight='bold')
plt.xlabel('Componente principal 1')
plt.ylabel('Componente principal 2')
plt.legend()
plt.show()


### Interpretacion de clusters
Se revisan los promedios por cluster para describir sus principales caracteristicas.


In [ ]:
perfil_clusters = data_segmentacion.groupby('cluster')[variables_segmentacion].mean().round(2)
tamano_clusters = data_segmentacion['cluster'].value_counts().sort_index()

display(perfil_clusters)
display(tamano_clusters)


In [ ]:
referencia = data_segmentacion[variables_segmentacion].median()

def nombrar_cluster(fila):
    if fila['precipitation'] >= referencia['precipitation'] and fila['relative_humidity_2m'] >= referencia['relative_humidity_2m']:
        return 'Lluvioso y humedo'
    if fila['sunshine_duration'] >= referencia['sunshine_duration'] and fila['cloud_cover'] <= referencia['cloud_cover']:
        return 'Despejado y soleado'
    if fila['wind_speed_10m'] >= referencia['wind_speed_10m']:
        return 'Ventoso'
    if fila['temperature_2m'] >= referencia['temperature_2m']:
        return 'Templado o calido'
    return 'Fresco o variable'

perfil_clusters['segmento'] = perfil_clusters.apply(nombrar_cluster, axis=1)
perfil_clusters.insert(0, 'cluster', perfil_clusters.index)
perfil_clusters


### Recomendaciones de negocio
A partir de los segmentos encontrados se proponen tres acciones simples y aplicables.


1. Priorizar alertas y planes preventivos en los segmentos lluviosos y humedos para reducir riesgos operativos.
2. Aprovechar los segmentos despejados y soleados para planificar actividades que dependen de buena visibilidad y baja precipitacion.
3. Usar los segmentos ventosos o variables para ajustar recursos, transporte y programacion de tareas que sean sensibles al viento.


### Cierre
Con esto se completa la fase de segmentacion solicitada en la pauta.
